In [2]:
using Revise
using Pkg;
#Pkg.develop(path="/home/gert/Projects/FusionRings.jl/")
#Pkg.develop(path="/Users/gertvercleyen/Projects/FusionRings.jl/")
using FusionRings
using Oscar
using JSON
using Base.Threads

In [2]:
function samefield_characters( r )
	to_composite_field( characters( r ), simplify_field = true )
end

samefield_characters (generic function with 1 method)

In [3]:
samefield_characters( frl[12] )

(AbsSimpleNumFieldElem[1 _a _a _a + 1; 1 _a -_a + 1 -1; 1 -_a + 1 _a -1; 1 -_a + 1 -_a + 1 -_a + 2], Map: number field -> QQBar)

In [4]:
function export_characters( i::Int ) 
	fn1 = "/home/gert/Tests/characters/chars_"* string(i) *".mrdi"
	fn2 = "/home/gert/Tests/characters/chars_injection_"* string(i) *".mrdi"

	function export_new_chars(i) 
		try 
			chars, f  = samefield_characters(frl[i])
			generator = gen( parent( chars[1] ) )
			Oscar.save( fn1, chars )
			Oscar.save( fn2, f(generator) )
		catch e2
			Oscar.save( fn1, ZZ.( [ 0 ] ) )
			Oscar.save( fn2, ZZ.( [ 0 ] ) )
		end	
	end
	
	try
		chars = Oscar.load(fn1)
		f     = Oscar.load(fn2)
		if chars == ZZ.([0])
			export_new_chars(i)
		end
	catch e
		export_new_chars(i)
	end
end

export_characters (generic function with 1 method)

In [5]:
function create_ind()
	indices = []
	fn(i) = "/home/gert/Tests/characters/chars_injection_"* string(i) *".mrdi"
	for j in 1:352
        if !FusionRings.is_commutative(frl[j])
            continue
        else
    		try 
    			f = Oscar.load(fn(j))
    			if f == ZZ.([0])
    				push!( indices, j )
    			end
    			continue
    		catch e
    			push!( indices, j )
    		end
        end
	end
	indices
end

create_ind (generic function with 1 method)

In [6]:
reverse(create_ind())

15-element Vector{Any}:
 346
 300
 282
 281
 273
 254
 241
 229
 228
 201
 181
 180
 149
 107
  88

In [ ]:
@threads for i in create_ind()
    export_characters(i)
    print(" * ")
end

2×2 Matrix{QQBarFieldElem}:
 {a1: 1.00}  {a1: 1.00}
 {a1: 1.00}  {a1: -1.00}

In [9]:
qqbar(1)

{a1: 1.00000}

# Finding Characters By Solving System of equations

In [55]:
function solve_character_equations( ring )
    r    = rank(ring)
    R, χ = polynomial_ring( QQ, :χ => 1:r )  
    m    = multiplication_table(ring) 
    setunit( pol ) = evaluate( pol, [χ[1]], [R(1)] )
    I = 
        ideal( 
            R, 
            unique(
            vec(
                [ 
                    setunit( χ[i]*χ[j] - sum( m[i,j,k]*χ[k] for k in 1:r ) ) 
                    for i in 2:r, j in 2:r 
                ]
            ))
        )
    groebner_basis(I,complete_reduction = true)
end

solve_character_equations (generic function with 1 method)

In [57]:
solve_character_equations( frl[300] )

Gröbner basis with elements
  1: χ[9]^2 - χ[2] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  2: χ[8]*χ[9] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9] - 1
  3: χ[7]*χ[9] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  4: χ[6]*χ[9] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  5: χ[5]*χ[9] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  6: χ[4]*χ[9] - χ[6] - χ[7] - χ[8] - χ[9]
  7: χ[3]*χ[9] - χ[7] - χ[8] - χ[9]
  8: χ[2]*χ[9] - χ[8]
  9: χ[8]^2 - χ[2] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  10: χ[7]*χ[8] - χ[3] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  11: χ[6]*χ[8] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  12: χ[5]*χ[8] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  13: χ[4]*χ[8] - χ[6] - χ[7] - χ[8] - χ[9]
  14: χ[3]*χ[8] - χ[7] - χ[8] - χ[9]
  15: χ[2]*χ[8] - χ[9]
  16: χ[7]^2 - χ[2] - χ[4] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9] - 1
  17: χ[6]*χ[7] - χ[3] - χ[5] - χ[6] - χ[7] - χ[8] - χ[9]
  18: χ[5]*χ[7] - χ[4] - χ[6] - χ[7] - χ[8] - χ[9]
  19: χ[4]*χ[7] - χ[5] - χ[7] - χ[8] - χ[9]

In [8]:
frl[3]

LoadError: UndefVarError: `frl` not defined in `Main`
Suggestion: check for spelling errors or missing imports.